# PourCastAI — Step 5: Risk APIs -> Bronze Delta

Same pattern as Step 4 (HubSpot), applied to the 3 live risk APIs that
risk_tools.py already calls locally: OSRM (routing), NWS (road hazards,
statewide), Open-Meteo (per-store forecast). Diesel (EIA) is deliberately
NOT here -- it stays on its existing local n8n cache path, see Step 6.

**Before running:** upload store_coords.csv (from export_store_coords.py)
into a Volume: Catalog -> pourcastai -> Create Volume (call it "landing")
-> Upload to this volume -> select store_coords.csv.

In [0]:
CATALOG = "pourcastai"
SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/pourcastai/bronze/landing/store_coords.csv"   # adjust if your volume name differs

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

stores_df = spark.read.csv(VOLUME_PATH, header=True, inferSchema=True)
stores = [row.asDict() for row in stores_df.collect()]
print(f"Loaded {len(stores)} store coordinates.")

Loaded 19 store coordinates.


In [0]:
import requests
import time
from datetime import datetime

TIMEOUT = 8
HEADERS = {"User-Agent": "PourCastAI-student-project (contact: example@uni.edu)"}
ANKENY_LAT, ANKENY_LONG = 41.699, -93.558
ingested_at = datetime.utcnow().isoformat()

# ----------------------------------------------------------------------------
# 1. OSRM -- one route per store (Ankeny -> store)
# ----------------------------------------------------------------------------
osrm_rows = []
for s in stores:
    url = (f"https://router.project-osrm.org/route/v1/driving/"
           f"{ANKENY_LONG},{ANKENY_LAT};{s['longitude']},{s['latitude']}?overview=false")
    try:
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        routes = r.json().get("routes")
        leg = routes[0] if routes else None
        osrm_rows.append({
            "store_number": s["store_number"],
            "distance_km": (leg["distance"] / 1000) if leg else None,
            "duration_hr": (leg["duration"] / 3600) if leg else None,
            "is_live": leg is not None,
            "ingested_at": ingested_at,
        })
    except Exception:
        osrm_rows.append({"store_number": s["store_number"], "distance_km": None,
                           "duration_hr": None, "is_live": False, "ingested_at": ingested_at})
    time.sleep(0.1)   # polite pacing -- avoids hammering OSRM's public demo server

print(f"OSRM: {sum(r['is_live'] for r in osrm_rows)}/{len(osrm_rows)} routes live.")

/home/spark-f39de645-add6-4959-85d8-88/.ipykernel/87/command-6054788108246451-3449488764:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingested_at = datetime.utcnow().isoformat()


OSRM: 19/19 routes live.


In [0]:
# ----------------------------------------------------------------------------
# 2. NWS road hazards -- ONE statewide call, same as risk_tools.py
# ----------------------------------------------------------------------------
ROAD_HAZARD_EVENTS = [
    "Winter Storm Warning", "Winter Weather Advisory", "Ice Storm Warning",
    "Dense Fog Advisory", "High Wind Warning", "Blizzard Warning", "Flood Warning",
]
try:
    r = requests.get("https://api.weather.gov/alerts/active?area=IA", timeout=TIMEOUT, headers=HEADERS)
    feats = r.json().get("features", [])
    relevant = [f for f in feats if f["properties"].get("event") in ROAD_HAZARD_EVENTS]
    hazard_row = [{"hazard_count": len(relevant),
                   "severity": min(len(relevant) / 5.0, 1.0),
                   "is_live": True, "ingested_at": ingested_at}]
except Exception:
    hazard_row = [{"hazard_count": 0, "severity": 0.0, "is_live": False, "ingested_at": ingested_at}]

print(f"NWS road hazards: {hazard_row[0]['hazard_count']} active statewide.")

NWS road hazards: 0 active statewide.


In [0]:
# ----------------------------------------------------------------------------
# 2b. NWS point alerts -- per-store, this is the 25%-weighted factor in
# risk_agent.py's score_route(). Distinct from road hazards above (statewide,
# 10%): this checks each store's own location for active severe alerts.
# ----------------------------------------------------------------------------
SEVERITY_RANK = {"Minor": 0.3, "Moderate": 0.6, "Severe": 0.85, "Extreme": 1.0}

alert_rows = []
for s in stores:
    url = f"https://api.weather.gov/alerts/active?point={s['latitude']},{s['longitude']}"
    try:
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        feats = r.json().get("features", [])
        sev = max([SEVERITY_RANK.get(f["properties"].get("severity"), 0.2) for f in feats], default=0.0)
        alert_rows.append({"store_number": s["store_number"], "alert_count": len(feats),
                            "severity": sev, "is_live": True, "ingested_at": ingested_at})
    except Exception:
        alert_rows.append({"store_number": s["store_number"], "alert_count": 0,
                            "severity": 0.0, "is_live": False, "ingested_at": ingested_at})
    time.sleep(0.1)

print(f"NWS point alerts: {sum(r['is_live'] for r in alert_rows)}/{len(alert_rows)} live, "
      f"{sum(r['alert_count'] for r in alert_rows)} total active alerts.")

NWS point alerts: 19/19 live, 0 total active alerts.


In [0]:
# ----------------------------------------------------------------------------
# 3. Open-Meteo -- one forecast per store
# ----------------------------------------------------------------------------
weather_rows = []
for s in stores:
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={s['latitude']}&longitude={s['longitude']}"
           "&hourly=precipitation_probability,wind_speed_10m&forecast_days=1")
    try:
        r = requests.get(url, timeout=TIMEOUT)
        hourly = r.json()["hourly"]
        precip = max(hourly["precipitation_probability"][:12], default=0) / 100.0
        wind = max(hourly["wind_speed_10m"][:12], default=0.0)
        weather_rows.append({
            "store_number": s["store_number"], "precip_prob": precip, "wind_kph": wind,
            "is_live": True, "ingested_at": ingested_at,
        })
    except Exception:
        weather_rows.append({"store_number": s["store_number"], "precip_prob": 0.0,
                              "wind_kph": 0.0, "is_live": False, "ingested_at": ingested_at})
    time.sleep(0.1)

print(f"Open-Meteo: {sum(r['is_live'] for r in weather_rows)}/{len(weather_rows)} forecasts live.")

Open-Meteo: 19/19 forecasts live.


In [0]:
# ----------------------------------------------------------------------------
# Write all 3 to Bronze
# ----------------------------------------------------------------------------
spark.createDataFrame(osrm_rows).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.osrm_routes")
spark.createDataFrame(hazard_row).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.nws_road_hazards")
spark.createDataFrame(alert_rows).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.nws_point_alerts")
spark.createDataFrame(weather_rows).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.weather_forecast")

print("\nBronze risk tables written:")
print(f"  {CATALOG}.{SCHEMA}.osrm_routes        ({len(osrm_rows)} rows)")
print(f"  {CATALOG}.{SCHEMA}.nws_road_hazards    ({len(hazard_row)} row)")
print(f"  {CATALOG}.{SCHEMA}.nws_point_alerts    ({len(alert_rows)} rows)")
print(f"  {CATALOG}.{SCHEMA}.weather_forecast    ({len(weather_rows)} rows)")


Bronze risk tables written:
  pourcastai.bronze.osrm_routes        (19 rows)
  pourcastai.bronze.nws_road_hazards    (1 row)
  pourcastai.bronze.nws_point_alerts    (19 rows)
  pourcastai.bronze.weather_forecast    (19 rows)


In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.osrm_routes ORDER BY distance_km"))

distance_km,duration_hr,ingested_at,is_live,store_number
16.0973,0.26672222222222225,2026-08-19T15:19:41.772947,true,2190
16.4358,0.2867222222222222,2026-08-19T15:19:41.772947,true,2627
16.6293,0.29133333333333333,2026-08-19T15:19:41.772947,true,4829
19.7834,0.33172222222222225,2026-08-19T15:19:41.772947,true,2633
25.5773,0.3652222222222222,2026-08-19T15:19:41.772947,true,3420
36.5867,0.4900555555555556,2026-08-19T15:19:41.772947,true,3814
39.6694,0.5020833333333333,2026-08-19T15:19:41.772947,true,3524
144.4973,1.8346666666666667,2026-08-19T15:19:41.772947,true,2593
164.68079999999998,2.2370277777777776,2026-08-19T15:19:41.772947,true,3494
176.14329999999998,2.2538333333333336,2026-08-19T15:19:41.772947,true,3773
